# TP 4 / 10 — Évaluation actuarielle du modèle

**M2 Actuariat — Abidjan**

---

## Objectifs
- Comprendre pourquoi **l'accuracy seule est trompeuse** en classification déséquilibrée.
- Connaître les métriques **data science** classiques : matrice de confusion, précision, rappel, F1, ROC-AUC, courbe précision-rappel.
- Maîtriser les métriques **actuarielles** : **courbe de lift**, **courbe de Lorenz**, **indice de Gini**.
- Diagnostiquer la **calibration** d'un modèle de tarification (Brier score, courbe de calibration).

> En tarification, un modèle qui **classe correctement les assurés du plus risqué au moins risqué** est plus important qu'un modèle qui a une bonne accuracy. C'est ce que mesure **le Gini / la courbe de lift**.

**Durée estimée : ~45 min.**

---


## 0. Imports et chargement des probabilités GLM du TP 3


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    brier_score_loss,
)

sns.set_theme(style="whitegrid")

y_test = pd.read_csv("y_test.csv").squeeze("columns")
y_train = pd.read_csv("y_train.csv").squeeze("columns")
proba_test  = pd.read_csv("proba_glm_test.csv").squeeze("columns")
proba_train = pd.read_csv("proba_glm_train.csv").squeeze("columns")

print(f"Test  : {len(y_test)} obs, taux BAD réel = {y_test.mean():.3%}")
print(f"Proba test  : min={proba_test.min():.3f}, moy={proba_test.mean():.3f}, max={proba_test.max():.3f}")


## 1. Évaluation au seuil 0.5 — le piège de l'accuracy


In [ ]:
y_pred = (proba_test >= 0.5).astype(int)

print("Matrice de confusion (seuil 0.5) :")
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["GOOD prédit", "BAD prédit"],
            yticklabels=["GOOD réel", "BAD réel"], ax=ax)
ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=["GOOD", "BAD"], digits=3))

acc = (y_pred == y_test).mean()
acc_naif = max(y_test.mean(), 1 - y_test.mean())
print(f"Accuracy du modèle : {acc:.3%}")
print(f"Accuracy du modèle naïf 'tout GOOD' : {acc_naif:.3%}")


### Question 1
- Au seuil 0.5, combien de **vrais BAD** notre modèle a-t-il identifiés ? Combien en a-t-il **manqués** (faux négatifs) ?
- Comparez l'accuracy du modèle à celle du « tout GOOD ». Le modèle apporte-t-il un gain visible ?
- Pourquoi un rappel très bas sur la classe BAD est-il **inacceptable en assurance** ?

*Votre réponse :*


## 2. Courbe ROC et AUC

La ROC est **invariante au seuil** : elle juge la capacité du modèle à classer correctement un BAD avant un GOOD, sur **toutes** les valeurs de seuil possibles.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, proba_test)
auc = roc_auc_score(y_test, proba_test)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="steelblue", label=f"GLM (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Modèle aléatoire")
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR / rappel)")
plt.title("Courbe ROC — GLM logistique sur le test")
plt.legend(loc="lower right")
plt.show()

print(f"AUC = {auc:.4f}")
print(f"  - Interprétation : sur un BAD et un GOOD tirés au hasard,")
print(f"    le modèle attribue une proba plus élevée au BAD dans {auc:.1%} des cas.")


## 3. Courbe précision-rappel (PR)

Plus informative que la ROC quand la classe positive est rare.


In [ ]:
prec, rec, _ = precision_recall_curve(y_test, proba_test)
ap = average_precision_score(y_test, proba_test)

plt.figure(figsize=(6, 5))
plt.plot(rec, prec, color="darkorange", label=f"GLM (AP = {ap:.3f})")
plt.axhline(y_test.mean(), color="gray", linestyle="--",
            label=f"Modèle aléatoire (= taux BAD = {y_test.mean():.2%})")
plt.xlabel("Rappel")
plt.ylabel("Précision")
plt.title("Courbe précision-rappel")
plt.legend()
plt.show()


### Question 2
- L'AUC vaut ~0.65–0.75. Est-ce un « bon » modèle pour de la tarification IARD ? (Comparez avec ce que vous avez pu voir en cours.)
- Pourquoi la **précision moyenne (AP)** est-elle plus parlante que l'AUC dans notre cas ?

*Votre réponse :*


## 4. Métrique actuarielle clé n°1 — la courbe de lift par déciles

**Principe** : on trie les contrats du test du plus au moins risqué selon la probabilité prédite, on les découpe en **10 groupes (déciles)**, et on regarde le taux de sinistralité réel dans chaque décile.

- Si le modèle est bon, le **décile 1 (les 10% jugés les plus risqués)** doit concentrer un taux de BAD **bien supérieur** à la moyenne, et le décile 10 (les moins risqués) doit en avoir très peu.
- Le **lift du décile 1** = (taux BAD dans décile 1) / (taux BAD global). Un lift de 3 signifie : « les 10% les plus risqués selon le modèle ont 3× plus de sinistres que la moyenne du portefeuille ».

C'est l'outil n°1 du tarificateur pour évaluer le **pouvoir discriminant** d'un modèle.


In [ ]:
def lift_table(y_true, y_proba, n_bins=10):
    df = pd.DataFrame({"y": y_true.values, "p": y_proba.values})
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decile"] = pd.qcut(df.index, q=n_bins, labels=False) + 1
    out = df.groupby("decile").agg(
        n=("y", "size"),
        n_bad=("y", "sum"),
        taux_bad=("y", "mean"),
        proba_moy=("p", "mean"),
    )
    out["lift"] = out["taux_bad"] / df["y"].mean()
    out["%_bad_cum"]  = out["n_bad"].cumsum() / out["n_bad"].sum()
    out["%_pop_cum"]  = out["n"].cumsum()    / out["n"].sum()
    return out

lt = lift_table(y_test, proba_test, n_bins=10)
lt.style.format({
    "taux_bad": "{:.2%}", "proba_moy": "{:.2%}", "lift": "{:.2f}",
    "%_bad_cum": "{:.1%}", "%_pop_cum": "{:.1%}",
})


In [ ]:
# Visualisation : lift par décile + lift cumulé
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(lt.index, lt["lift"], color="steelblue", edgecolor="k")
axes[0].axhline(1.0, color="red", linestyle="--", label="Lift = 1 (modèle nul)")
axes[0].set_xlabel("Décile de risque (1 = 10% les + risqués)")
axes[0].set_ylabel("Lift")
axes[0].set_title("Lift par décile")
axes[0].legend()

axes[1].plot([0] + list(lt["%_pop_cum"]), [0] + list(lt["%_bad_cum"]),
             marker="o", color="steelblue", label="GLM")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="Modèle aléatoire")
axes[1].set_xlabel("% du portefeuille (trié par risque décroissant)")
axes[1].set_ylabel("% des sinistres capturés")
axes[1].set_title("Courbe de lift cumulée (Gain Chart)")
axes[1].legend()

plt.tight_layout()
plt.show()


### Question 3 — application métier
- En examinant **uniquement les 20% des contrats les plus risqués** d'après le modèle, quel pourcentage de l'ensemble des sinistres BAD attrape-t-on ?
- Si la direction souhaite faire de la **prévention ciblée** (appel téléphonique, audit) sur un budget de 10% du portefeuille, le modèle l'aide-t-il à allouer ce budget intelligemment ? Quantifiez le gain par rapport à un tirage aléatoire.
- Quelle est la **valeur business** d'un lift de 3 sur le décile 1 ?

*Votre réponse :*


## 5. Métrique actuarielle clé n°2 — courbe de Lorenz et indice de Gini

La courbe de Lorenz et le **coefficient de Gini** sont historiquement issus de l'économie (mesure d'inégalités), repris massivement en tarification. Définition :

$$ \mathrm{Gini} = 2 \times \mathrm{AUC} - 1 \in [0, 1]. $$

- Gini = 0 : modèle non discriminant.
- Gini = 1 : modèle parfait.
- Gini = 0.30 – 0.50 : ordre de grandeur typique d'un modèle de fréquence en assurance auto.

La courbe de Lorenz trace le **% cumulé de sinistres capturés** en fonction du **% cumulé d'exposition** (ici, % cumulé du portefeuille). L'aire entre la courbe et la diagonale, multipliée par 2, donne le Gini.


In [ ]:
# Calcul exact (Gini = 2*AUC - 1 quand l'exposition est constante par contrat)
gini = 2 * auc - 1
print(f"AUC = {auc:.4f}  →  Gini = {gini:.4f}")

# Courbe de Lorenz tracée à partir des contrats triés par proba décroissante
order = np.argsort(-proba_test.values)
y_sorted = y_test.values[order]
cum_pop  = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
cum_bad  = np.cumsum(y_sorted) / y_sorted.sum()

plt.figure(figsize=(6, 6))
plt.plot([0] + list(cum_pop), [0] + list(cum_bad),
         color="steelblue", label=f"GLM (Gini = {gini:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Modèle aléatoire")
plt.fill_between([0] + list(cum_pop), [0] + list(cum_pop), [0] + list(cum_bad),
                 color="steelblue", alpha=0.15)
plt.xlabel("% cumulé du portefeuille")
plt.ylabel("% cumulé de sinistres BAD")
plt.title("Courbe de Lorenz — l'aire bleue × 2 = Gini")
plt.legend()
plt.show()


### Question 4
- Démontrer (en une ligne) que **Gini = 2·AUC − 1** quand toutes les expositions sont identiques.
- Un modèle de fréquence d'assurance auto en France affiche en général Gini ∈ [0.30, 0.45]. Le nôtre est-il dans cet intervalle ?
- Si on doublait la taille du portefeuille **sans changer les caractéristiques**, le Gini changerait-il ? Et la courbe de Lorenz ?

*Votre réponse :*


## 6. Calibration — le modèle est-il honnête sur ses probabilités ?

L'AUC et le Gini mesurent l'**ordonnancement** (« est-ce qu'on trie bien ? »).
La **calibration** mesure si les probabilités prédites correspondent aux fréquences observées :
- quand le modèle dit « 10% de chance d'être BAD », observe-t-on bien 10% de BAD ?
- crucial en tarification : la prime pure est proportionnelle à $\hat\pi$, donc une mauvaise calibration sur-tarife ou sous-tarife.

Outils :
- **Brier score** : $\frac{1}{n}\sum (y_i - \hat\pi_i)^2$. Plus c'est petit, mieux c'est.
- **Courbe de calibration** : on bin les probas prédites, on compare proba moyenne vs fréquence observée.


In [ ]:
# Brier score
brier = brier_score_loss(y_test, proba_test)
print(f"Brier score GLM : {brier:.4f}")
print(f"Brier score modèle constant (proba = taux moyen) : "
      f"{brier_score_loss(y_test, np.full_like(y_test, y_train.mean(), dtype=float)):.4f}")


In [ ]:
def calibration_curve_custom(y_true, y_proba, n_bins=10):
    df = pd.DataFrame({"y": y_true.values, "p": y_proba.values})
    df["bin"] = pd.qcut(df["p"], q=n_bins, duplicates="drop")
    out = df.groupby("bin", observed=True).agg(
        proba_moy=("p", "mean"),
        freq_obs =("y", "mean"),
        n        =("y", "size"),
    )
    return out

cc = calibration_curve_custom(y_test, proba_test, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot(cc["proba_moy"], cc["freq_obs"], marker="o", color="steelblue", label="GLM")
plt.plot([0, cc["proba_moy"].max()*1.1], [0, cc["proba_moy"].max()*1.1],
         "--", color="gray", label="Calibration parfaite")
plt.xlabel("Probabilité moyenne prédite")
plt.ylabel("Fréquence observée de BAD")
plt.title("Courbe de calibration (par décile de proba)")
plt.legend()
plt.show()

print(cc.round(4))


### Question 5
- La courbe de calibration suit-elle bien la diagonale ? Si le modèle s'écarte, est-il **sur-confiant** (proba prédite > fréquence observée) ou **sous-confiant** ?
- Un GLM logistique standard est-il **structurellement bien calibré** sur le jeu d'entraînement (rappel cours) ? Et sur le test ?
- Pourquoi un modèle peut-il avoir un **bon Gini** mais une **mauvaise calibration** ? Donnez un exemple.

*Votre réponse :*


## 7. Synthèse pour les prochains TP

Récapitulatif des résultats sur le test :


In [ ]:
resume = pd.Series({
    "Accuracy seuil 0.5": (y_pred == y_test).mean(),
    "Recall BAD seuil 0.5": ((y_pred == 1) & (y_test == 1)).sum() / (y_test == 1).sum(),
    "AUC": auc,
    "Gini": gini,
    "Average Precision": ap,
    "Brier": brier,
    "Lift décile 1": lt.loc[1, "lift"],
}).round(4)
print(resume)

resume.to_csv("metrics_glm_baseline.csv", header=["valeur"])
print("\nSauvegardé : metrics_glm_baseline.csv")


### Question 6 — synthèse
- Sur **quelle métrique** le GLM brille-t-il le plus ? Sur laquelle est-il le plus faible ?
- Si vous deviez **améliorer le rappel sur la classe BAD**, sans changer le modèle, quelle action mécanique pouvez-vous faire ? (Indice : c'est le sujet du TP 5.)
- Le TP 7 introduira un LightGBM. Pour qu'il « gagne », quelle métrique devrait-il améliorer ?

*Votre réponse :*

---
**Prochain TP : TP 5 — GLM amélioré (interactions, régularisation, choix de seuil).**

On essaiera de **dépasser** les performances du baseline mesurées ici.
